# Design 1: Piston Pump — living calculations

Python is the active model. This notebook uses the same hydraulic equations
as the archived reference; it does not add a controller, actuator or Windkessel.
See `../MODEL.md` and `../VARIABLES.md` for assumptions and the variable register.
All pressures in `Case` are gauge except explicitly labelled absolute quantities.
A sealed air pocket uses absolute pressure in its gas law.

In [ ]:
from pathlib import Path
import sys
from dataclasses import asdict, replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

here = Path.cwd()
candidates = [here, here.parent, here / "prototypes/v1/design-1-piston-pump"]
root = next((p for p in candidates if (p / "python/hydraulic_sweep.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open from the notebook directory, design directory or repository root.")
sys.path.insert(0, str(root / "python"))
from hydraulic_sweep import Case, simulate


## Prescribed targets and design inputs

Edit the Case below to define an operating test. Changing diameter while
keeping swept volume fixed changes the required travel. Target flow is a
requirement for comparison, not an additional independent boundary imposed
on the hydraulic ODE.

In [ ]:
flow_target_Lmin = 5.0
case = Case(f=75.0, diameter_mm=50.0, swept_ml=5000/75,
            valve_in_mm=16.0, valve_out_mm=16.0, gas_ref_ml=10.0)
area_mm2 = np.pi * case.diameter_mm**2 / 4
travel_mm = 1000 * case.swept_ml / area_mm2
required_net_ml = 1000 * flow_target_Lmin / case.f
print(f"Travel: {travel_mm:.3f} mm; required net delivery: {required_net_ml:.3f} mL/beat")
pd.Series(asdict(case), name="Case value")

## Variable types

Design parameters, operating inputs, boundary conditions, constitutive
properties, dynamic states and calculated outputs have different physical
roles. The CSV additionally records Python types, units, defaults and whether
the variable is implemented. Future measurements and control commands are
not silently treated as existing model inputs.

In [ ]:
variables = pd.read_csv(root / "VARIABLES.csv")
print(f"{len(variables)} register entries including future interfaces and numerical diagnostics")
variables[["group", "symbol", "meaning", "role", "python_type", "units", "status"]]

## Fixed stroke and a prescribed mean flow

Mean circuit flow = delivery frequency × net delivered volume per beat / 1000.
The following ideal volume budget is separate from the dynamic solution.

In [ ]:
bpm = np.array([40, 60, 75, 90, 120, 150, 180], dtype=float)
required_ml = 1000 * flow_target_Lmin / bpm
ideal_flow = bpm * case.swept_ml / 1000
fraction = required_ml / case.swept_ml
budget = pd.DataFrame({"bpm": bpm, "ideal_full_stroke_Lmin": ideal_flow,
    "required_net_mL": required_ml,
    "ideal_diverted_fraction": np.where(fraction <= 1, 1-fraction, np.nan)})
display(budget)
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(bpm, ideal_flow, "o-", label="Ideal full stroke")
ax.axhline(flow_target_Lmin, linestyle="--", label="Target")
ax.set(xlabel="Deliveries/min", ylabel="Mean flow (L/min)")
ax.grid(alpha=.25); ax.legend(); plt.show()

## New baseline simulation

The next cell executes the current Python model and shows its actual delivered
flow and signed hydraulic force. These are uncalibrated predictions. A guard
or non-periodic result is not displayed as a valid periodic solution.

In [ ]:
result = simulate(case, keep_trace=True)
print("Status:", result["status"])
if result["status"] != "periodic":
    raise RuntimeError("This case did not produce a valid periodic result: " + result["status"])
trace = result["trace"]
display(pd.Series({k:result[k] for k in ["Q_Lmin", "delivered_ml", "force_absmax_N",
    "pc_peak_mmhg", "pc_min_mmhg", "fill_deficit_ml", "balance_error_ml"]}))
fig, axes = plt.subplots(2,1,figsize=(8,6),sharex=True,layout="constrained")
axes[0].plot(trace["time_s"], trace["pressure_force_N"])
axes[0].set_ylabel("Signed pressure force (N)")
axes[1].plot(trace["time_s"], trace["qin_Lmin"], label="Inlet")
axes[1].plot(trace["time_s"], trace["qout_Lmin"], label="Outlet")
axes[1].set(xlabel="Time (s)", ylabel="Flow (L/min)")
axes[1].legend()
for ax in axes: ax.grid(alpha=.25)
plt.show()

## Archived parameter sweep

The table below is the previously executed 118-case reference, not a fresh
sweep from this notebook. Run `python python/run_model.py sweep` from the design
directory to regenerate the full sweep and figures into `results/python/`.

In [ ]:
archive = pd.read_csv(root / "archive/pump_hydraulic_sweep/case_results.csv")
display(archive.groupby("status").size().rename("Case count"))
archive[["f", "valve_in_mm", "gas_ref_ml", "status", "force_absmax_N", "Q_Lmin"]].head(12)